# Assignment 3. Frequency Domain Processing

<span style="color:orange">Ground Rules for the Assignment: </span>

* <span style="color:lightblue"> You can only use basic functions (matrix operations, input/output image functions, plotters). Anything else, you need to code from scratch (histogram functions, inverting gamma functions, color matting, histogram equalization)</span>
* <span style="color:lightblue"> The code needs to be appropiately commented and should be reproducible; if we cannot re-generate your figures from your code, we will deduct points.</span>
* <span style="color:lightblue">The notebook report should be detailed and include partial and final solutions for each exercise. We grade solely the report; code without report will not be graded, so we encourage that you invest some time on it</span>
* <span style="color:lightblue">Interactive plots are welcome but most important results should be static and generated beforehand</span>
* <span style="color:lightblue">__Remember to remove all plots from the "coding" sections.__ Only the Report should output plots and/or images.</span>


<span style="color:orange">Submission Details</span>

Simply submit this Jupyter Notebook with the report inlined as described below. The notebook should be executed before submission. Name the file as 
```surname1_name1_surname2_name2_assignment3.ipynb```


In [ ]:
# Loading Libraries you will need for the assignment.- Install them in your environment if you haven't done so yet
import numpy as np
from numpy import fft
import matplotlib.pyplot as plt
from PIL import Image
import cv2 #opencv python
from matplotlib.axes import Axes
import plotly.express as px

# 0. Helpers

In [ ]:
def load_rgb_image(path: str, max_size: tuple[int, int] | None = None) -> Image.Image:
    """Load an image in RGB mode.

    A maximum size can optionally be provided to keep the notebook lightweight and
    reproducible when the original files are very large.
    """
    image = Image.open(path).convert("RGB")
    if max_size is not None:
        image.thumbnail(max_size, Image.Resampling.LANCZOS)
    return image


def image2array(image: Image.Image) -> np.ndarray:
    """Convert a PIL image to a float32 NumPy array in [0, 1]."""
    image = np.array(image)
    image = image.astype(np.float32) / 255.0
    return image


def array2image(image: np.ndarray) -> Image.Image:
    """Convert a float image in [0, 1] back to an 8-bit PIL image."""
    image = np.clip(image * 255, 0, 255).astype(np.uint8)
    return Image.fromarray(image)


def display(
        image: Image.Image | np.ndarray, gamma: float | None = None, title: str | None = None, ax: Axes | None = None
):
    """Display an image.

    gamma=None (or 0) means the image is already in display space.
    A positive gamma value is applied before display, which is useful for visualizing
    linear images.
    """
    if isinstance(image, Image.Image):
        image = image2array(image)

    if gamma not in (None, 0):
        image = np.clip(image, 0, 1) ** gamma

    if ax is not None:
        ax.imshow(np.clip(image, 0, 1))
        ax.set_title(title)
        ax.axis("off")
        return

    plt.imshow(np.clip(image, 0, 1))
    plt.title(title)
    plt.axis("off")

## 1. Gaussian Filtering [10 points]
In this exercise we will be working on filtering and the connection between spatial and frequency domain filtering.
### _Tasks_

* Implement Gaussian filtering both in the spatial and frequency domains
* Demonstrate that convolving an image with a Gaussian filter with standard deviation $\sigma_s$ in the spatial domain is equivalent to point-wise multiplication in the frequency domain with Gaussian filter with standard deviation $\sigma_f = \frac{1}{2\sigma{s}\pi}$
* Analyze how the performance of equivalent filtering in spatial and temporal domains depends on the parameter $\sigma_s$

![image.png](attachment:image.png)

In [ ]:
# Define your functions here .- you can include comments in your code explaining the steps of your algorithm

In [ ]:
# Execute your code here

###  <span style="color:orange"> _Report_ </span>
<span style="color:orange"> _Report your results below. As a test image for this exercise, create an image similar to the one shown in Figure 1. For filtering both spatial and frequency domains assume padding with zero values. In the report, please show examples of filtered images with different pairs of $\sigma_s$ and $\sigma_f$. In particular, include in your report
a plot of the execution time for both domains as a function of $\sigma_s$. Please inline the resulting images with your text explaining the approach (e.g. as figures) so that the report is cohesive._ </span>

(your report)

## 2. Image Restoration [10 points]
Consider a task of removing a repetitive pattern from an image using filtering in the frequency domain. Figure 2 demonstrates an input and the corresponding output of such a procedure. 
### _Tasks_
Design and implement a filtering procedure which perform such restoration. Explain your technique, show Fourier plots of all the steps, as well as the final image. Use the input image provided with the assignment.


![image.png](attachment:image.png)

In [ ]:
def create_gaussian_notch(shape, center, sigma=10.0):
    """
    Generates a 2D Gaussian notch reject filter mask.

    Parameters:
        shape (tuple): The (rows, cols) of the frequency domain image.
        center (tuple): The (row, col) coordinates of the noise spike.
        sigma (float): The spread of the Gaussian curve (larger = wider cut).

    Returns:
        np.ndarray: A 2D array of floats ranging from 0.0 (at the center) to 1.0.
    """
    rows, cols = shape
    center_row, center_col = center

    # Generate efficient broadcastable coordinate grids
    y, x = np.ogrid[:rows, :cols]

    # Calculate the squared Euclidean distance from the given center
    dist_sq = (x - center_col)**2 + (y - center_row)**2

    # Apply the Gaussian formula: H(u,v) = 1 - e^(-D^2 / 2*sigma^2)
    mask = 1.0 - np.exp(-dist_sq / (2.0 * sigma**2))

    return mask

In [ ]:
san_img = load_rgb_image("./san_domenico.png")
san_arr = image2array(san_img)
san_arr = np.pad(san_arr, pad_width=50, mode='symmetric')
rows, cols = san_arr.shape[:2]

ft = fft.fft2(san_arr.mean(axis=2))
ft = fft.fftshift(ft)
ft_abs = abs(ft)
ft_norm = np.log1p(ft_abs)
plt.imshow(ft_norm)

In [ ]:

fig = px.imshow(
    ft_norm,
    color_continuous_scale='Viridis',  # You can use 'gray' for standard black/white
    title="Interactive 2D Function",
    labels=dict(x="X-axis", y="Y-axis", color="Intensity")
)

fig.show()

In [ ]:
centers = [
    (210, 280),
    (215, 353),
]
length = 10
r = 10

masked_ft = ft.copy()
im_center = (rows//2, cols//2)
for center in centers:
    sim_center = (cols - center[0], rows - center[1])
    for y, x in [center, sim_center]:
        masked_ft[x, y-length:y+length] = 0
        masked_ft[x-length:x+length, y] = 0
        masked_ft[x-r:x+r, y-r:y+r] = 0

ft_abs = abs(masked_ft)
ft_norm = np.log1p(ft_abs)

fig = px.imshow(
    ft_norm,
    color_continuous_scale='Viridis', # You can use 'gray' for standard black/white
    title="Interactive 2D Function",
    labels=dict(x="X-axis", y="Y-axis", color="Intensity")
)

fig.show()

In [ ]:
plt.imshow(ft_norm)

In [ ]:
img_restore = fft.ifft2(fft.ifftshift(masked_ft))
display(np.real(img_restore))

In [ ]:
centers = [
    (254, 324),
    (260, 332),
    (254, 416),
    (260, 408),
]

r = 10

masked_ft = ft.copy()
im_center = (rows//2, cols//2)
for center in centers:
    sim_center = (cols - center[0], rows - center[1])
    for y, x in [center, sim_center]:
        mask = create_gaussian_notch(ft.shape, (x, y), sigma=r)
        masked_ft = masked_ft * mask

ft_abs = abs(masked_ft)
ft_norm = np.log1p(ft_abs)

fig = px.imshow(
    ft_norm,
    color_continuous_scale='Viridis', # You can use 'gray' for standard black/white
    title="Interactive 2D Function",
    labels=dict(x="X-axis", y="Y-axis", color="Intensity")
)

fig.show()

In [ ]:
img_restore = fft.ifft2(fft.ifftshift(masked_ft))
display(np.real(img_restore[50:-50, 50:-50]))

###  <span style="color:orange"> _Report_ </span>
<span style="color:orange"> _Report your results below. Explain your technique, show Fourier plots of all the steps, as well as the final image. Use the input image provided with the assignment. Please inline the resulting images with your text explaining the approach (e.g. as figures) so that the report is cohesive._ </span>

(your report)